# CWRU bearing signals: healthy vs inner-race fault

We compare `97.mat` (normal, 0 HP) and `105.mat` (12k drive-end, 0.007 inch inner-race fault, 0 HP). The CWRU pages list the normal 0 HP speed as about 1797 rpm and describe drive-end recordings at 12 kHz and 48 kHz. The normal-baseline table does not identify the sample rate of `97.mat` per file, so these first plots use sample index. Verify that file's rate before using seconds or interpreting frequencies.

Sources: [CWRU normal baseline](https://engineering.case.edu/bearingdatacenter/normal-baseline-data), [CWRU 12k drive-end data](https://engineering.case.edu/bearingdatacenter/12k-drive-end-bearing-fault-data), [CWRU download notes](https://engineering.case.edu/bearingdatacenter/download-data-file).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy.io import loadmat

from motorsense_ml.features import crest_factor, kurtosis, peak, rms

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "ml" / "data").is_dir())
DATA_DIR = ROOT / "ml" / "data"
FILES = {"healthy": DATA_DIR / "97.mat", "inner-race fault": DATA_DIR / "105.mat"}
WINDOW_SAMPLES = 4096

signals = {}
for label, path in FILES.items():
    mat = loadmat(path)
    keys = [key for key in mat if key.endswith("_DE_time")]
    if len(keys) != 1:
        raise ValueError(f"Expected one *_DE_time variable in {path.name}; found {keys}")
    signal = np.asarray(mat[keys[0]]).reshape(-1)
    signals[label] = (keys[0], signal)
    print(f"{label}: {keys[0]}, {signal.size:,} samples")

In [ ]:
fig, axes = plt.subplots(len(signals), 1, figsize=(12, 6), sharex=True, constrained_layout=True)
for ax, (label, (_, signal)) in zip(axes, signals.items(), strict=False):
    shown = signal[:WINDOW_SAMPLES]
    ax.plot(np.arange(shown.size), shown, linewidth=0.7)
    ax.set_title(label)
    ax.set_ylabel("Acceleration (recorded units)")
axes[-1].set_xlabel("Sample index (sampling rate not applied)")
plt.show()

In [ ]:
feature_rows = []
for label, (_, signal) in signals.items():
    window = signal[:WINDOW_SAMPLES]
    feature_rows.append(
        {
            "condition": label,
            "rms": rms(window),
            "peak": peak(window),
            "crest_factor": crest_factor(window),
            "kurtosis": kurtosis(window),
        }
    )

for row in feature_rows:
    print(row)